# 01 · Carga del dataset weatherAUS

En este notebook cargamos el dataset **Rain in Australia** de Kaggle y hacemos una primera revisión: tamaño, tipos de datos y valores nulos. Todavía no se limpia nada, solo se mira cómo vienen los datos.

## 1. Librerías

Importamos pandas y las funciones que dejamos en `src/carga_datos.py`. Agregamos la carpeta del proyecto al path para poder importar `src`.

In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

raiz = Path.cwd().resolve()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.append(str(raiz))

from src.carga_datos import cargar_weather, validar_columnas, resumen_nulos

np.random.seed(42)
print("Carpeta del proyecto:", raiz)

Carpeta del proyecto: C:\MCDIA\mcdi501-estadistica-computacional


## 2. Cargar los datos

El CSV está en `data/raw/weatherAUS.csv`. Usamos `parse_dates` para que la columna `Date` quede como fecha y `na_values=['NA']` porque así vienen marcados los faltantes en este archivo.

In [5]:
datos = cargar_weather()
print("Filas:", datos.shape[0])
print("Columnas:", datos.shape[1])
datos.head()

Filas: 145460
Columnas: 23


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## 3. Validación básica

Revisamos que estén las 23 columnas originales y que la fecha se haya leído bien. Usamos `assert` para que el notebook se detenga si algo no cuadra.

In [6]:
assert validar_columnas(datos)
assert datos.shape[1] == 23, "El dataset deberia tener 23 columnas"
assert pd.api.types.is_datetime64_any_dtype(datos["Date"]), "Date no quedo como fecha"
print("Validaciones OK")
print("Rango de fechas:", datos["Date"].min().date(), "a", datos["Date"].max().date())
print("Cantidad de ciudades:", datos["Location"].nunique())

Validaciones OK
Rango de fechas: 2007-11-01 a 2017-06-25
Cantidad de ciudades: 49


## 4. Tipos de datos

Vemos qué columnas son numéricas y cuáles son texto (categóricas).

In [7]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           145460 non-null  datetime64[us]
 1   Location       145460 non-null  str           
 2   MinTemp        143975 non-null  float64       
 3   MaxTemp        144199 non-null  float64       
 4   Rainfall       142199 non-null  float64       
 5   Evaporation    82670 non-null   float64       
 6   Sunshine       75625 non-null   float64       
 7   WindGustDir    135134 non-null  str           
 8   WindGustSpeed  135197 non-null  float64       
 9   WindDir9am     134894 non-null  str           
 10  WindDir3pm     141232 non-null  str           
 11  WindSpeed9am   143693 non-null  float64       
 12  WindSpeed3pm   142398 non-null  float64       
 13  Humidity9am    142806 non-null  float64       
 14  Humidity3pm    140953 non-null  float64       
 15  Pressure9am

## 5. Estadísticas descriptivas

Primero las variables numéricas y después las categóricas.

In [8]:
datos.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
Date,145460,2013-04-04 21:08:51.907053,2007-11-01 00:00:00,2011-01-11 00:00:00,2013-06-02 00:00:00,2015-06-14 00:00:00,2017-06-25 00:00:00,NaN
MinTemp,143975.0,12.194034,-8.5,7.6,12.0,16.9,33.9,6.398495
MaxTemp,144199.0,23.221348,-4.8,17.9,22.6,28.2,48.1,7.119049
Rainfall,142199.0,2.360918,0.0,0.0,0.0,0.8,371.0,8.47806
Evaporation,82670.0,5.468232,0.0,2.6,4.8,7.4,145.0,4.193704
Sunshine,75625.0,7.611178,0.0,4.8,8.4,10.6,14.5,3.785483
WindGustSpeed,135197.0,40.03523,6.0,31.0,39.0,48.0,135.0,13.607062
WindSpeed9am,143693.0,14.043426,0.0,7.0,13.0,19.0,130.0,8.915375
WindSpeed3pm,142398.0,18.662657,0.0,13.0,19.0,24.0,87.0,8.8098
Humidity9am,142806.0,68.880831,0.0,57.0,70.0,83.0,100.0,19.029164


In [9]:
categoricas = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday", "RainTomorrow"]
datos[categoricas].describe().T

,count,unique,top,freq
Location,145460,49,Canberra,3436
WindGustDir,135134,16,W,9915
WindDir9am,134894,16,N,11758
WindDir3pm,141232,16,SE,10838
RainToday,142199,2,No,110319
RainTomorrow,142193,2,No,110316


## 6. Valores nulos

Contamos cuántos faltantes hay por columna. Esto sirve para decidir después qué hacer con ellos.

In [10]:
nulos = resumen_nulos(datos)
nulos

,nulos,porcentaje
Sunshine,69835,48.01
Evaporation,62790,43.17
Cloud3pm,59358,40.81
Cloud9am,55888,38.42
Pressure9am,15065,10.36
Pressure3pm,15028,10.33
WindDir9am,10566,7.26
WindGustDir,10326,7.10
WindGustSpeed,10263,7.06
Humidity3pm,4507,3.10


## 7. Variable objetivo

Miramos cómo se reparte `RainTomorrow` (si llueve al día siguiente).

In [11]:
datos["RainTomorrow"].value_counts(dropna=False)

RainTomorrow
No     110316
Yes     31877
NaN      3267
Name: count, dtype: int64

In [12]:
(datos["RainTomorrow"].value_counts(normalize=True) * 100).round(2)

RainTomorrow
No     77.58
Yes    22.42
Name: proportion, dtype: float64

## 8. Comentarios

- El dataset se cargó con ruta relativa desde `data/raw/`.